# Puzzle Metadata Extraction

**Primary author:** Victoria Winters

**Builds on:**
- *DATA_RAW.md* (shared reference — §4 source-by-source extraction logic, lookup table semantics, and edge-case catalog)
- *data/publisher_lookup.csv* (curated by Victoria — 349 rows mapping normalized extracted substrings to canonical publisher / series / setter)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Extracts structured puzzle metadata from `data/clues_raw.csv` and writes `data/puzzle_metadata.csv`, a standalone one-row-per-`clue_id` artifact shared across all project components. Columns produced: `publisher`, `series`, `setter`, `puzzle_no`, `puzzle_date`, `clue_no`, `clue_direction`. The grain is per-`clue_id` (not per-puzzle) because `clue_no` and `clue_direction` vary within a single puzzle; the puzzle-level columns are repeated across all clues belonging to the same puzzle. No `puzzle_id` column is introduced — that design decision is explicitly deferred.

---

## §0 — Imports and Paths

All paths are relative to this notebook's directory via `pathlib`. This notebook is Local only — no environment detection needed.

In [ ]:
# === Imports and Paths ===
import re
import time
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data"
RAW_PATH = DATA_DIR / "clues_raw.csv"
LOOKUP_PATH = DATA_DIR / "publisher_lookup.csv"
OUTPUT_PATH = DATA_DIR / "puzzle_metadata.csv"

---

## §1 — Load Data

Two loads. From `clues_raw.csv` we pull only the six columns this notebook needs to derive puzzle metadata: `clue_id`, `source`, `puzzle_name`, `source_url`, `puzzle_date`, and `clue_number`. `keep_default_na=False` with `na_values=[""]` is required because the word `nan` (a valid crossword entry meaning "grandmother") would otherwise be silently coerced to `NaN` — the same flag used in `structural_filtering.ipynb`.

The second load brings in `publisher_lookup.csv`, the curated mapping from normalized extracted substrings (in its `raw` column) to canonical `publisher`, `series`, and `setter` values. Its `field` column identifies which extraction location each row applies to (`puzzle_name_leading`, `puzzle_name_trailing`, or `source_url_trailing`).

After loading, a per-source row count is printed so any drift in the raw input is visible at a glance.

In [ ]:
# === Load Data ===
t0 = time.time()
df = pd.read_csv(
    RAW_PATH,
    usecols=[
        "clue_id",
        "source",
        "puzzle_name",
        "source_url",
        "puzzle_date",
        "clue_number",
    ],
    keep_default_na=False,
    na_values=[""],
)
load_runtime = time.time() - t0
print(f"Loaded {len(df):,} rows from {RAW_PATH} in {load_runtime:.1f}s")
print("  (keep_default_na=False protects crossword entries like 'nan' "
      "from being coerced to NaN)")
print()

lookup = pd.read_csv(LOOKUP_PATH)
print(f"Loaded {len(lookup):,} rows from {LOOKUP_PATH}")
print("  Curated mapping from normalized extracted substrings to "
      "canonical publisher / series / setter")
print()

print("Per-source row counts in clues_raw.csv:")
print(df["source"].value_counts().to_string())

---

## §2 — Parse `clue_number`

The raw `clue_number` column encodes a grid position as a digit run followed by `a` (across) or `d` (down), e.g. `"23a"`, `"18d"`. A single regex splits each value into two derived columns: `clue_no` (integer) and `clue_direction` (`"across"` or `"down"`). Anything that doesn't match — empty strings, stray punctuation, or unusual formats — yields NaN for both columns.

Unparseable values are counted and a small sample printed so unusual formats don't get silently swallowed.

In [ ]:
# === Parse clue_number ===
extracted = df["clue_number"].astype("string").str.extract(
    r"^\s*(\d+)\s*([adAD])\s*$"
)
df["clue_no"] = pd.to_numeric(extracted[0], errors="coerce").astype("Int64")
df["clue_direction"] = (
    extracted[1].str.lower().map({"a": "across", "d": "down"})
)

# A row is unparseable if it had a raw clue_number value but the regex
# failed to split it into (digits, a/d).
unparseable_mask = df["clue_number"].notna() & df["clue_no"].isna()
n_unparseable = int(unparseable_mask.sum())
print(f"Unparseable clue_number values: {n_unparseable:,}")

sample = (
    df.loc[unparseable_mask, "clue_number"]
    .drop_duplicates()
    .head(5)
    .tolist()
)
print(f"Sample of unparseable raw strings: {sample}")

---

## §3 — Extract `puzzle_name_leading`

`puzzle_name_leading` is the normalized text before the first run of digits in `puzzle_name`. It is the primary lookup key for four sources:

- `bigdave44` — publisher + series (e.g. `"Toughie 2766"` → `toughie`)
- `fifteensquared` — publisher + series (e.g. `"Guardian Cryptic 27509 by Qaos"` → `guardian`)
- `times_xwd_times` — series only; publisher is hardcoded in §4
- `thehinducrosswordcorner` — series prefix; publisher is hardcoded in §4

The normalization is designed to produce a string comparable to the `raw` column in `publisher_lookup.csv` (lowercase, punctuation-stripped, free of the noise tokens `cryptic` and trailing `no`). It is applied across the whole dataframe, but the lookup itself is only performed for the four relevant sources — other sources don't use this field and we don't want their raw tokens to accidentally collide with one of the four sources' entries in the lookup table.

While we have `puzzle_name` parsed, we also extract `puzzle_no` from it in the same pass. Puzzle numbers are 3–5 digits, optionally preceded by `–`, `No`, or `No.`, sometimes with a stray trailing comma (`"4835,"`) or an embedded comma (`"26,495"`) — both forms need cleaning before casting to `int`. The column is stored as `Int64` (nullable integer) so unextractable rows can hold NA without being coerced to float.

After the merge, three diagnostics are printed: the number of rows that matched the lookup, the number of rows in the four relevant sources that did not match (these will become the unmatched-token log in Part 2), and a sample of up to 10 unmatched raw tokens per source. A similar breakdown is printed for `puzzle_no` extraction failures.

In [ ]:
# === Extract puzzle_name_leading ===
LEADING_SOURCES = [
    "bigdave44",
    "fifteensquared",
    "times_xwd_times",
    "thehinducrosswordcorner",
]

# Punctuation characters stripped from the ends of the extracted token.
# Includes ASCII hyphen, en-dash, em-dash, plus common brackets/quotes.
_END_PUNCT = r"\s\-\u2013\u2014.,:;!?'\"()\[\]"
_end_strip_re = re.compile(rf"^[{_END_PUNCT}]+|[{_END_PUNCT}]+$")
_collapse_ws_re = re.compile(r"\s+")
_pre_digit_re = re.compile(r"^(.*?)\d")


def normalize_leading(name):
    """Normalize the substring of ``name`` before its first digit run
    into the same shape used by the ``raw`` column of
    ``publisher_lookup.csv``.

    Returns ``None`` if ``name`` is not a string. An empty string is a
    meaningful result (e.g. the ``thehinducrosswordcorner`` main series
    where ``puzzle_name`` starts with ``\"No <digits>\"``).
    """
    if not isinstance(name, str):
        return None
    # 1. Text before the first digit sequence.
    m = _pre_digit_re.match(name)
    token = m.group(1) if m else name
    # 2. Lowercase.
    token = token.lower()
    # 3. Strip leading/trailing whitespace and punctuation (dashes included).
    token = _end_strip_re.sub("", token)
    # 4. Remove the noise word "cryptic" anywhere it appears as a
    #    standalone token, not as part of a compound like "cryptics".
    token = re.sub(r"(?:^|\s)cryptic(?=\s|$)", " ", token)
    token = _collapse_ws_re.sub(" ", token).strip()
    # 5. Strip a standalone trailing "no" (the puzzle-number prefix word).
    token = re.sub(r"(?:^|\s)no$", "", token).strip()
    return token


df["puzzle_name_leading_raw"] = df["puzzle_name"].apply(normalize_leading)

# Only look up the four relevant sources; blank the raw token elsewhere so
# unrelated sources can't accidentally collide with a lookup entry.
is_leading_source = df["source"].isin(LEADING_SOURCES)
df.loc[~is_leading_source, "puzzle_name_leading_raw"] = pd.NA

leading_lookup = (
    lookup[lookup["field"] == "puzzle_name_leading"]
    [["source", "raw", "publisher", "series", "setter"]]
    .drop_duplicates(subset=["source", "raw"])
    .rename(columns={"raw": "puzzle_name_leading_raw"})
)

df = df.merge(
    leading_lookup,
    on=["source", "puzzle_name_leading_raw"],
    how="left",
    indicator="_leading_merge",
)
matched_mask = df["_leading_merge"] == "both"
n_matched = int(matched_mask.sum())
print(f"Lookup matches on puzzle_name_leading: {n_matched:,}")

unmatched_mask = is_leading_source & ~matched_mask
n_unmatched = int(unmatched_mask.sum())
print(f"Unmatched rows in the four relevant sources: {n_unmatched:,}")

print()
print("Top unmatched raw tokens per source (up to 10 each):")
for src in LEADING_SOURCES:
    src_tokens = df.loc[
        unmatched_mask & (df["source"] == src), "puzzle_name_leading_raw"
    ]
    counts = src_tokens.value_counts(dropna=False).head(10)
    print(f"\n[{src}] {len(src_tokens):,} unmatched rows:")
    if counts.empty:
        print("  (none)")
    else:
        for tok, cnt in counts.items():
            print(f"  {cnt:>6,}  {tok!r}")

df = df.drop(columns=["_leading_merge"])

# ── Puzzle number extraction ────────────────────────────────────────
# Match either a comma-grouped number ("26,495") or a bare 3–5 digit run;
# the first alternative is tried first so it wins when a comma is present.
_puzzle_no_re = re.compile(r"(\d{1,3},\d{3}|\d{3,5})")


def extract_puzzle_no(name):
    """Return the first puzzle number embedded in ``name`` as an int,
    stripping any embedded or stray trailing comma. Returns ``pd.NA``
    if ``name`` is not a string or contains no matching digit run.
    """
    if not isinstance(name, str):
        return pd.NA
    m = _puzzle_no_re.search(name)
    if not m:
        return pd.NA
    return int(m.group(1).replace(",", ""))


df["puzzle_no"] = pd.Series([pd.NA] * len(df), dtype="object")
df.loc[is_leading_source, "puzzle_no"] = (
    df.loc[is_leading_source, "puzzle_name"].apply(extract_puzzle_no)
)
df["puzzle_no"] = pd.to_numeric(df["puzzle_no"], errors="coerce").astype("Int64")

print()
print("puzzle_no extraction failures per source:")
for src in LEADING_SOURCES:
    src_mask = df["source"] == src
    fail_mask = src_mask & df["puzzle_no"].isna()
    n_fail = int(fail_mask.sum())
    print(f"\n[{src}] {n_fail:,} rows with no extractable puzzle_no")
    if n_fail:
        sample = (
            df.loc[fail_mask, "puzzle_name"]
            .dropna()
            .drop_duplicates()
            .head(5)
            .tolist()
        )
        for s in sample:
            print(f"  {s!r}")

---

## §4 — Extract `puzzle_name_trailing`

`puzzle_name_trailing` is the text after the puzzle number (and, for `thehinducrosswordcorner`, after the date) in `puzzle_name`. Two sources use it as their setter lookup key:

- **`fifteensquared`** — the setter name appears after the puzzle number,   often preceded by the keyword `"by"` (e.g. `"Financial Times 16795 by   Artexlen"`), sometimes directly (`"Guardian 28378 Tramp"`), and   sometimes absent entirely (`"Everyman 3857"`). Per DATA_RAW.md §5,   month names bleed through as fake setter names from Sloggers & Betters   date-labelled entries; these are nullified via a blocklist.
- **`thehinducrosswordcorner`** — the setter is the third comma-separated   field, after the number and the date (e.g. `"No 13251, Wednesday 19   May 2021, Incognito"`). Sunday Crossword rows lack the setter field   and map to NaN. Punctuation variants of common setter names   (`"Dr. X,"`, `"Spinner,"`, etc.) are handled partly by stripping   trailing punctuation here and partly by dedicated rows in   `publisher_lookup.csv`.

Both extractions are done in a single pass; the lookup is a filtered view of `publisher_lookup.csv` keyed on `(source, raw)` with `field == "puzzle_name_trailing"`. Collaborative setters are stored pipe-separated in the lookup table and are not split further here.

After merging, newly-located setter values are coalesced into the existing `setter` column (which §3 left mostly NaN for these two sources). Match rates and samples of unmatched raw tokens are printed per source.

In [ ]:
# === Extract puzzle_name_trailing ===
_MONTH_BLOCKLIST = {
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
}
_after_first_number_re = re.compile(r"\d[\d,]*\s*(.*?)\s*$")
_trailing_punct_re = re.compile(r"[\s.,;:!?'\"()\-\u2013\u2014]+$")
_leading_by_re = re.compile(r"^by\s+", re.IGNORECASE)


def normalize_fifteensquared_trailing(name):
    """Normalize the fifteensquared trailing token for lookup.

    Grabs everything after the first number run in ``name``, strips an
    optional leading ``by`` keyword, lowercases, strips trailing
    punctuation, and nullifies month-name bleed-through. Returns
    ``None`` if no trailing token exists.
    """
    if not isinstance(name, str):
        return None
    m = _after_first_number_re.search(name)
    if not m:
        return None
    token = m.group(1)
    token = _leading_by_re.sub("", token).strip()
    token = token.lower()
    token = _trailing_punct_re.sub("", token)
    token = _collapse_ws_re.sub(" ", token).strip()
    if not token:
        return None
    if token in _MONTH_BLOCKLIST:
        return None
    return token


def normalize_hindu_trailing(name):
    """Normalize the thehinducrosswordcorner trailing token for lookup.

    Splits ``name`` on commas and returns the last field (lowercased,
    trailing-punctuation stripped) only when there are at least three
    comma-separated fields — i.e. the standard ``No N, Day DD Mon YYYY,
    Setter`` format. Sunday-Crossword rows without a setter return None.
    """
    if not isinstance(name, str):
        return None
    parts = [p.strip() for p in name.split(",")]
    if len(parts) < 3:
        return None
    token = parts[-1].lower()
    token = _trailing_punct_re.sub("", token)
    token = _collapse_ws_re.sub(" ", token).strip()
    return token or None


df["puzzle_name_trailing_raw"] = pd.NA
is_fs = df["source"] == "fifteensquared"
is_hindu = df["source"] == "thehinducrosswordcorner"
df.loc[is_fs, "puzzle_name_trailing_raw"] = (
    df.loc[is_fs, "puzzle_name"].apply(normalize_fifteensquared_trailing)
)
df.loc[is_hindu, "puzzle_name_trailing_raw"] = (
    df.loc[is_hindu, "puzzle_name"].apply(normalize_hindu_trailing)
)

trailing_lookup = (
    lookup[lookup["field"] == "puzzle_name_trailing"]
    [["source", "raw", "setter"]]
    .drop_duplicates(subset=["source", "raw"])
    .rename(columns={"raw": "puzzle_name_trailing_raw",
                     "setter": "setter_from_trailing"})
)

df = df.merge(
    trailing_lookup,
    on=["source", "puzzle_name_trailing_raw"],
    how="left",
    indicator="_trailing_merge",
)

# Coalesce the new setter value into the existing column. The trailing
# lookup is authoritative for fifteensquared and hindu; §3 left these
# mostly NaN (with a handful of leading-token exceptions like
# "guardian prize picaroon" that should be preserved).
df["setter"] = df["setter"].fillna(df["setter_from_trailing"])

trailing_matched = df["_trailing_merge"] == "both"
for src in ["fifteensquared", "thehinducrosswordcorner"]:
    src_mask = df["source"] == src
    extracted_mask = src_mask & df["puzzle_name_trailing_raw"].notna()
    matched = int((extracted_mask & trailing_matched).sum())
    total = int(extracted_mask.sum())
    pct = (matched / total * 100) if total else 0.0
    print(f"[{src}] trailing matches: {matched:,}/{total:,} "
          f"({pct:.1f}% of rows with an extracted token)")

    unmatched_tokens = (
        df.loc[extracted_mask & ~trailing_matched, "puzzle_name_trailing_raw"]
        .value_counts()
        .head(10)
    )
    if not unmatched_tokens.empty:
        print(f"  Top unmatched trailing tokens:")
        for tok, cnt in unmatched_tokens.items():
            print(f"    {cnt:>5,}  {tok!r}")
    print()

df = df.drop(columns=["setter_from_trailing", "_trailing_merge"])

---

## §5 — Extract `source_url_trailing`

`source_url_trailing` is the text at the end of `source_url` just before the file extension. Only `thebrowser` uses it, and only for setter lookup. Per DATA_RAW.md §4.3, the `source_url` format is `"thebrowser/Cryptic NN - MonDDYY SetterSurname.puz"` — the setter is whatever appears after the compressed `MonDDYY` date and before `.puz`. Collaborative puzzles end in two space-separated surnames (e.g. `"Jacobs Goodchild"`) which are kept intact and looked up as a single raw token; the lookup table stores the resolved value as a pipe-separated string (e.g. `"Jacobs|Goodchild"`). Stray leading dashes (`"- Ries"`) are preserved in the raw token because a dedicated lookup row handles them.

Two more fields are extracted from `thebrowser` rows in the same pass:

- `puzzle_no` — from `puzzle_name` via `#(\d+)` (format:   `"CRYPTIC #13 (March 27, 2021)"`)
- `puzzle_date` — from the parenthesized date in `puzzle_name`. The raw   `puzzle_date` column is essentially all-NaN for this source, so this   fills the gap. The result is stored as an ISO `YYYY-MM-DD` string to   match the format used by every other source in `clues_raw.csv`.

Match rate and a sample of unmatched url tokens are printed after the merge.

In [ ]:
# === Extract source_url_trailing ===
is_browser = df["source"] == "thebrowser"

# Capture everything after the MonDDYY date stamp and before .puz.
# MonDDYY is three-letter-month + 4 digits (DDYY), e.g. "Mar2721".
_browser_setter_re = re.compile(
    r"[A-Z][a-z]{2}\d{4}\s+([^./]+)\.puz$"
)


def extract_browser_url_trailing(url):
    """Pull the setter token from a thebrowser source_url.

    Returns the lowercased substring between the compressed date and
    the ``.puz`` extension, preserving internal spaces (for
    collaborative puzzles) and any stray leading dash. Returns None
    when the URL does not match the expected format.
    """
    if not isinstance(url, str):
        return None
    m = _browser_setter_re.search(url)
    if not m:
        return None
    return m.group(1).strip().lower()


df["source_url_trailing_raw"] = pd.NA
df.loc[is_browser, "source_url_trailing_raw"] = (
    df.loc[is_browser, "source_url"].apply(extract_browser_url_trailing)
)

url_lookup = (
    lookup[lookup["field"] == "source_url_trailing"]
    [["source", "raw", "setter"]]
    .drop_duplicates(subset=["source", "raw"])
    .rename(columns={"raw": "source_url_trailing_raw",
                     "setter": "setter_from_url"})
)

df = df.merge(
    url_lookup,
    on=["source", "source_url_trailing_raw"],
    how="left",
    indicator="_url_merge",
)
df["setter"] = df["setter"].fillna(df["setter_from_url"])

url_matched = df["_url_merge"] == "both"
extracted_browser = is_browser & df["source_url_trailing_raw"].notna()
n_browser_matched = int((extracted_browser & url_matched).sum())
n_browser_total = int(extracted_browser.sum())
pct = (n_browser_matched / n_browser_total * 100) if n_browser_total else 0.0
print(f"[thebrowser] source_url matches: {n_browser_matched:,}/"
      f"{n_browser_total:,} ({pct:.1f}% of rows with an extracted token)")

browser_unmatched = (
    df.loc[extracted_browser & ~url_matched, "source_url_trailing_raw"]
    .value_counts()
    .head(10)
)
if not browser_unmatched.empty:
    print("  Top unmatched url-trailing tokens:")
    for tok, cnt in browser_unmatched.items():
        print(f"    {cnt:>5,}  {tok!r}")

df = df.drop(columns=["setter_from_url", "_url_merge"])

# ── thebrowser puzzle_no and puzzle_date ────────────────────────────
_browser_num_re = re.compile(r"#(\d+)")
_browser_date_re = re.compile(r"\(([^)]+)\)\s*$")


def extract_browser_puzzle_no(name):
    if not isinstance(name, str):
        return pd.NA
    m = _browser_num_re.search(name)
    return int(m.group(1)) if m else pd.NA


def extract_browser_puzzle_date(name):
    """Extract and ISO-format the parenthesized date in a thebrowser
    puzzle_name (e.g. ``\"CRYPTIC #13 (March 27, 2021)\"``).
    """
    if not isinstance(name, str):
        return None
    m = _browser_date_re.search(name)
    if not m:
        return None
    parsed = pd.to_datetime(m.group(1), errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.strftime("%Y-%m-%d")


browser_puzzle_no = df.loc[is_browser, "puzzle_name"].apply(
    extract_browser_puzzle_no
)
df.loc[is_browser, "puzzle_no"] = browser_puzzle_no
df["puzzle_no"] = pd.to_numeric(df["puzzle_no"], errors="coerce").astype("Int64")

browser_dates = df.loc[is_browser, "puzzle_name"].apply(
    extract_browser_puzzle_date
)
df.loc[is_browser, "puzzle_date"] = df.loc[is_browser, "puzzle_date"].fillna(
    browser_dates
)

n_browser_no = int(df.loc[is_browser, "puzzle_no"].notna().sum())
n_browser_date = int(df.loc[is_browser, "puzzle_date"].notna().sum())
print(f"\n[thebrowser] puzzle_no extracted for {n_browser_no:,}/"
      f"{int(is_browser.sum()):,} rows")
print(f"[thebrowser] puzzle_date filled for {n_browser_date:,}/"
      f"{int(is_browser.sum()):,} rows")

---

## §6 — Hardcoded Assignments

Per DATA_RAW.md §4.1, several sources have `publisher` (and sometimes `setter`) values that are assigned directly in code rather than via the lookup table. This cell applies those assignments, plus a source-specific `puzzle_no` / `puzzle_date` rescue where needed:

- **`natpostcryptic`** — `publisher = "National Post"`,   `setter = "Hex"` (pseudonym of Emily Cox & Henry Rathvon).
- **`cru_cryptics`** — `publisher = "Cru Cryptics Forum"`; `puzzle_no`   from `source_url` via `Cryptic(\d+)\.puz`.
- **`nytimes`** — `publisher = "New York Times"`; `puzzle_date` from   `source_url` via an 8-digit date parsed as `%Y%m%d` (the raw   `puzzle_date` is NaN for this source).
- **`newyorker`** — `publisher = "The New Yorker"`; `puzzle_no` from   `puzzle_name` via `No\.\s*(\d+)` with a fallback to `no-(\d+)` in   `source_url` for date-titled entries.
- **`leoedit`** — `publisher = "Amuselabs Leoedit"`; nothing else   extractable.
- **`thebrowser`** — `publisher = "The Browser"` (setter already   populated in §5).
- **`thehinducrosswordcorner`** — `publisher = "The Hindu"` for every   row, regardless of whether the leading-token lookup matched (most   don't, since the main series has an empty leading token).
- **`times_xwd_times`** — publisher is not in the lookup at all. Assign   `"The Times"` when the §3 series lookup placed the row in a Times   series, `"Sunday Times"` for Sunday series, and   `"Times Literary Supplement"` for the TLS series. This must happen   after §3 because it reads from the already-populated `series` column.

After each assignment, the cell prints how many rows were affected per source so coverage is visible.

In [ ]:
# === Hardcoded Assignments ===
HARDCODED_PUBLISHER = {
    "natpostcryptic": "National Post",
    "cru_cryptics": "Cru Cryptics Forum",
    "nytimes": "New York Times",
    "newyorker": "The New Yorker",
    "leoedit": "Amuselabs Leoedit",
    "thebrowser": "The Browser",
    "thehinducrosswordcorner": "The Hindu",
}
HARDCODED_SETTER = {
    "natpostcryptic": "Hex",
}

for src, pub in HARDCODED_PUBLISHER.items():
    mask = df["source"] == src
    df.loc[mask, "publisher"] = df.loc[mask, "publisher"].fillna(pub)
    n = int(mask.sum())
    print(f"[{src}] publisher := {pub!r}  ({n:,} rows)")

for src, setter in HARDCODED_SETTER.items():
    mask = df["source"] == src
    df.loc[mask, "setter"] = df.loc[mask, "setter"].fillna(setter)
    n = int(mask.sum())
    print(f"[{src}] setter := {setter!r}  ({n:,} rows)")

# times_xwd_times — infer publisher from the series the §3 lookup assigned.
times_mask = df["source"] == "times_xwd_times"
times_series = df.loc[times_mask, "series"].fillna("")

sunday_series = {
    "Sunday Times Cryptic", "Mephisto", "Christmas Special",
    "Christmas Extra",
}
tls_series = {"TLS Crossword"}

publisher_by_series = times_series.map(
    lambda s: "Sunday Times" if s in sunday_series
    else "Times Literary Supplement" if s in tls_series
    else ("The Times" if s else pd.NA)
)
df.loc[times_mask, "publisher"] = (
    df.loc[times_mask, "publisher"].fillna(publisher_by_series)
)
print(
    f"[times_xwd_times] publisher assigned from series — "
    f"The Times: {(publisher_by_series == 'The Times').sum():,}, "
    f"Sunday Times: {(publisher_by_series == 'Sunday Times').sum():,}, "
    f"Times Literary Supplement: "
    f"{(publisher_by_series == 'Times Literary Supplement').sum():,}"
)

# cru_cryptics — puzzle_no from source_url.
cru_mask = df["source"] == "cru_cryptics"
cru_nums = df.loc[cru_mask, "source_url"].str.extract(r"Cryptic(\d+)\.puz")[0]
df.loc[cru_mask, "puzzle_no"] = pd.to_numeric(cru_nums, errors="coerce")
df["puzzle_no"] = pd.to_numeric(df["puzzle_no"], errors="coerce").astype("Int64")
print(
    f"[cru_cryptics] puzzle_no extracted from source_url for "
    f"{int(df.loc[cru_mask, 'puzzle_no'].notna().sum()):,}/"
    f"{int(cru_mask.sum()):,} rows"
)

# nytimes — puzzle_date from source_url 8-digit stamp.
nyt_mask = df["source"] == "nytimes"
nyt_dates_raw = df.loc[nyt_mask, "source_url"].str.extract(r"(\d{8})")[0]
nyt_dates = pd.to_datetime(nyt_dates_raw, format="%Y%m%d", errors="coerce")
df.loc[nyt_mask, "puzzle_date"] = df.loc[nyt_mask, "puzzle_date"].fillna(
    nyt_dates.dt.strftime("%Y-%m-%d")
)
print(
    f"[nytimes] puzzle_date extracted from source_url for "
    f"{int(df.loc[nyt_mask, 'puzzle_date'].notna().sum()):,}/"
    f"{int(nyt_mask.sum()):,} rows"
)

# newyorker — puzzle_no from puzzle_name, fallback to source_url.
ny_mask = df["source"] == "newyorker"
ny_name_nums = (
    df.loc[ny_mask, "puzzle_name"].str.extract(r"No\.\s*(\d+)")[0]
)
ny_url_nums = df.loc[ny_mask, "source_url"].str.extract(r"no-(\d+)")[0]
ny_combined = ny_name_nums.fillna(ny_url_nums)
df.loc[ny_mask, "puzzle_no"] = pd.to_numeric(ny_combined, errors="coerce")
df["puzzle_no"] = pd.to_numeric(df["puzzle_no"], errors="coerce").astype("Int64")
print(
    f"[newyorker] puzzle_no extracted for "
    f"{int(df.loc[ny_mask, 'puzzle_no'].notna().sum()):,}/"
    f"{int(ny_mask.sum()):,} rows"
)

---

## §7 — Merge and Finalize

Assemble the final output dataframe. The schema is fixed by the shared pipeline: `clue_id`, `publisher`, `series`, `setter`, `puzzle_no`, `puzzle_date`, `clue_no`, `clue_direction`, in that order. `clue_id` serves as the join key for downstream components; it is unique in this file (one row per raw clue) but becomes non-unique after multi-definition expansion in `clues_filtered.csv`.

`puzzle_no` and `clue_no` are stored as `Int64` (pandas nullable integer) so that missing values don't silently promote the whole column to float. `setter` is a plain pipe-separated string — collaborative entries like `"Enigmatist|Soup"` are kept in that form; downstream code that needs a list calls `.str.split("|")` itself.

Per-source coverage of the four derived columns is printed as a fraction of total rows in that source, so any unexpected gaps are immediately visible.

In [ ]:
# === Merge and Finalize ===
output = df[[
    "clue_id",
    "publisher",
    "series",
    "setter",
    "puzzle_no",
    "puzzle_date",
    "clue_no",
    "clue_direction",
]].copy()

output["puzzle_no"] = output["puzzle_no"].astype("Int64")
output["clue_no"] = output["clue_no"].astype("Int64")

print(f"Output shape: {output.shape}")
print()
print("dtypes:")
print(output.dtypes.to_string())
print()
print("Sample (10 rows, one per source where possible):")
sample = (
    output.assign(_src=df["source"])
    .groupby("_src", group_keys=False)
    .head(2)
    .drop(columns=["_src"])
    .head(10)
)
with pd.option_context("display.max_colwidth", 40, "display.width", 200):
    print(sample.to_string(index=False))
print()

print("Per-source coverage (non-NaN fraction):")
coverage_cols = ["publisher", "series", "setter", "puzzle_no", "puzzle_date"]
coverage = (
    output.assign(_src=df["source"])
    .groupby("_src")[coverage_cols]
    .apply(lambda g: g.notna().mean())
)
with pd.option_context("display.float_format", "{:.2%}".format):
    print(coverage.to_string())

---

## §8 — Unmatched Token Log

A required diagnostic: every raw token extracted from `puzzle_name` or `source_url` that was not found in `publisher_lookup.csv` is logged here, grouped by `(source, field)`. The log is the input to future lookup table maintenance — each unmatched token either represents a legitimate setter/series that should be added to the lookup, or noise (malformed entries, month-name bleed-through, blank puzzle names) that should be left as NaN in the final output.

In [ ]:
# === Unmatched Token Log ===
# Rebuild per-field match indicators by re-merging against each lookup
# subset. Doing this at the end keeps the §3–§5 cells focused on their
# primary task while still giving us a complete unmatched view here.
log_sources = {
    "puzzle_name_leading": (
        "puzzle_name_leading_raw",
        ["bigdave44", "fifteensquared", "times_xwd_times",
         "thehinducrosswordcorner"],
    ),
    "puzzle_name_trailing": (
        "puzzle_name_trailing_raw",
        ["fifteensquared", "thehinducrosswordcorner"],
    ),
    "source_url_trailing": (
        "source_url_trailing_raw",
        ["thebrowser"],
    ),
}

for field, (raw_col, field_sources) in log_sources.items():
    field_lookup_keys = set(
        zip(
            lookup.loc[lookup["field"] == field, "source"],
            lookup.loc[lookup["field"] == field, "raw"],
        )
    )
    for src in field_sources:
        sub = df.loc[df["source"] == src, raw_col]
        extracted = sub.dropna()
        unmatched = extracted[
            ~extracted.map(lambda r, s=src: (s, r) in field_lookup_keys)
        ]
        counts = unmatched.value_counts()
        print(f"── {src} / {field} "
              f"── {len(counts):,} unique unmatched tokens, "
              f"{int(unmatched.shape[0]):,} rows ──")
        if counts.empty:
            print("  (none)")
        else:
            for tok, cnt in counts.items():
                print(f"  {cnt:>6,}  {tok!r}")
        print()

### Analysis of unmatched tokens

*[TO FILL IN after running]* — After running the notebook, Victoria should discuss here: which `(source, field)` groups carry the largest unmatched counts, whether the tokens appear to be noise (`__trashed`, single letters, month names, blank strings, puzzle-title leakage) or legitimate setter / series names missing from the lookup, and what (if anything) should be added to `publisher_lookup.csv` as a follow-up. `thehinducrosswordcorner / puzzle_name_leading` is expected to carry a large empty-string bucket corresponding to the main series, which is already absorbed by the §6 hardcoded publisher assignment and is not a genuine unmatched case.

---

## §9 — Write Output

Write the finalized dataframe to `data/puzzle_metadata.csv`. This file lives in the shared `data/` directory and should **not** be regenerated without a documented reason agreed with the team — downstream components join against it on `clue_id`, and a silent change here could propagate into trained models.

In [ ]:
# === Write Output ===
# Shared upstream artifact: do not regenerate without a documented reason
# agreed with the team. See CLAUDE.md -> Shared Data Directory Rules.
output.to_csv(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"Wrote {len(output):,} rows to {OUTPUT_PATH} ({size_mb:.1f} MB)")

---

## §10 — Summary

This notebook turns the raw `puzzle_name`, `source_url`, `puzzle_date`, and `clue_number` columns of `clues_raw.csv` into a tidy per-`clue_id` metadata table and writes it to `data/puzzle_metadata.csv` — a shared upstream artifact consumed by every project component.

### What was done

1. **§1 — Load.** Read six columns from `clues_raw.csv`    (`clue_id`, `source`, `puzzle_name`, `source_url`, `puzzle_date`,    `clue_number`) with `keep_default_na=False, na_values=[""]` so the    crossword entry `"nan"` survives, and load the full    `publisher_lookup.csv`.
2. **§2 — Clue number.** Parse `clue_number` with a single regex into    `clue_no` (`Int64`) and `clue_direction` (`"across"` / `"down"`).
3. **§3 — Leading-token lookup.** Normalize the substring of    `puzzle_name` before its first digit run, look it up against    `publisher_lookup.csv` where `field == "puzzle_name_leading"` for    `bigdave44`, `fifteensquared`, `times_xwd_times`, and    `thehinducrosswordcorner`, and populate `publisher`, `series`, and    (occasionally) `setter`. Also extract `puzzle_no` from the first    3–5 digit run in `puzzle_name`, handling stray/embedded commas.
4. **§4 — Trailing-token lookup.** For `fifteensquared`, extract the    substring after the first number (stripping an optional `by`    keyword and month-name bleed from Sloggers & Betters entries) and    look up the setter. For `thehinducrosswordcorner`, the setter is    the third comma-separated field; Sunday Crossword rows that lack    this field become NaN. New setter values are coalesced into the    existing `setter` column.
5. **§5 — URL-trailing lookup.** For `thebrowser`, extract the    substring of `source_url` between the compressed `MonDDYY` date and    `.puz`, look up the setter, and additionally extract `puzzle_no`    from `#NN` in `puzzle_name` and reconstruct the mostly-missing    `puzzle_date` from the parenthesized date, formatted as ISO    `YYYY-MM-DD` to match the other sources.
6. **§6 — Hardcoded assignments.** Apply direct publisher (and, for    `natpostcryptic`, setter) assignments for sources that don't use    the lookup table at all. For `times_xwd_times`, derive publisher    from the series assigned in §3 (`The Times`, `Sunday Times`,    `Times Literary Supplement`). For `cru_cryptics`, `nytimes`, and    `newyorker`, extract `puzzle_no` or `puzzle_date` from `source_url`    / `puzzle_name` regexes as needed.
7. **§7–§9 — Assemble, diagnose, write.** Project to the fixed output    schema, print per-source column coverage, log every unmatched raw    token in §8, and write `data/puzzle_metadata.csv`.

### Output schema

`data/puzzle_metadata.csv` — one row per `clue_id`, columns in this order:

| column | type | notes |
|---|---|---|
| `clue_id` | int | Unique here; **not unique** in `clues_filtered.csv` after multi-definition expansion. |
| `publisher` | string | Canonical publisher name. |
| `series` | string | Puzzle series within the publication; NaN where not applicable. |
| `setter` | string | Pipe-separated for collaborations (e.g. `"Enigmatist|Soup"`); NaN where unknown. Never a Python list. |
| `puzzle_no` | `Int64` | Nullable integer; NaN where not extractable. |
| `puzzle_date` | string | ISO `YYYY-MM-DD`. |
| `clue_no` | `Int64` | Grid number parsed from raw `clue_number`. |
| `clue_direction` | string | `"across"` or `"down"`; NaN for unparseable rows. |

### Deferred design decision: no `puzzle_id`

A `puzzle_id` column (a unique key per puzzle, for grouping clues at the puzzle level) was considered and **deliberately not introduced**. The grain of this table is per-`clue_id`, and the puzzle-level columns (`publisher`, `series`, `setter`, `puzzle_no`, `puzzle_date`) are simply repeated across every clue in a puzzle. Downstream components that need a puzzle key can derive one on demand from `(publisher, series, puzzle_no, puzzle_date)`; committing to a specific composite or synthetic key is left as an open decision until a component actually needs it.

### Key edge cases

- **`"nan"` the crossword entry** — `keep_default_na=False` is load-  critical (§1); without it, rows whose definition or answer is the   word "nan" (meaning grandmother) get silently coerced to NaN.
- **`times_xwd_times` noise** — `puzzle_name` for this source carries   month-only tokens (`"April"`), puzzle-title leakage   (`"Red Scare,"`), single-letter entries (`"T"`), blank entries   (185 rows), stray trailing commas on puzzle numbers (`"4835,"`),   and embedded commas inside numbers (`"26,495"`). The leading-token   normalization and the puzzle-number regex (which allows an optional   inner comma) handle the first two; unresolvable tokens map to NaN   and land in §8.
- **`fifteensquared` month-name bleed** — Sloggers & Betters entries   can leave a bare month name as the trailing token; §4 nullifies any   token that matches the English month blocklist before the lookup.
- **`thehinducrosswordcorner` setter normalization** — `"Dr. X"`   appears in several punctuation variants (`"Dr. X,"`, `"Dr X"`,   `"Dr, X,"`); `"Anon"` maps to NaN (pre-2008 anonymous puzzles).   The trailing-punctuation strip in §4 plus the dedicated lookup rows   in `publisher_lookup.csv` handle these between them.
- **`thebrowser` date reconstruction** — raw `puzzle_date` is almost   entirely NaN for this source; §5 rebuilds it from the parenthesized   date in `puzzle_name` and stores the result as an ISO string for   format parity with every other source.

### Unmatched tokens

See **§8** for the full per-source, per-field log of raw tokens extracted but not matched against `publisher_lookup.csv`, and the analysis cell underneath it for the human interpretation.